# Hospital Readmission Prediction

Predict 30-day hospital readmission risk using Logistic Regression with L2 regularization.

**Features:** age, diagnosis, heart rate, systolic blood pressure, previous visits, and hospital stay duration.

**Evaluation:** ROC-AUC, confusion matrix, precision, recall, and clinical false-negative/false-positive analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, accuracy_score,
    precision_score, recall_score, ConfusionMatrixDisplay
)

## 1. Create a Sample Patient Dataset

In [ ]:
data = {
    'Age': [65,42,71,35,58,76,49,63,29,68,54,80,45,72,38,61,50,69,33,74,
            57,47,66,31,70,52,44,77,36,60,55,73,41,67,28,59,48,75,39,64],
    'Diagnosis': ['Diabetes','Infection','Heart Disease','Asthma','Diabetes',
                 'Heart Disease','Infection','Diabetes','Asthma','Heart Disease',
                 'Diabetes','Heart Disease','Infection','Diabetes','Asthma',
                 'Heart Disease','Diabetes','Heart Disease','Asthma','Diabetes',
                 'Infection','Asthma','Heart Disease','Asthma','Diabetes',
                 'Infection','Heart Disease','Diabetes','Asthma','Infection',
                 'Diabetes','Heart Disease','Infection','Diabetes','Asthma',
                 'Heart Disease','Infection','Diabetes','Asthma','Heart Disease'],
    'Heart_Rate': [95,78,105,82,91,110,85,98,76,108,94,115,88,102,80,107,96,112,79,100,
                  86,83,109,77,97,84,106,118,81,89,99,113,87,101,75,104,90,96,78,111],
    'Systolic_BP': [140,120,150,118,135,160,125,145,115,155,138,165,128,148,119,152,142,158,116,145,
                    130,122,151,117,143,126,154,168,121,132,146,162,124,149,114,156,129,141,118,159],
    'Previous_Visits': [4,1,5,0,3,6,2,4,0,5,3,7,1,4,0,5,3,6,0,4,2,1,5,0,4,2,6,7,1,3,4,5,2,4,0,5,2,3,1,6],
    'Stay_Days': [7,3,10,2,6,12,4,8,2,9,5,14,3,7,2,10,6,11,2,8,4,3,9,2,7,4,10,13,3,5,7,9,4,6,2,11,5,6,3,10],
    'Readmitted': [1,0,1,0,1,1,0,1,0,1,0,1,0,1,0,1,1,1,0,1,0,0,1,0,1,0,1,1,0,0,1,1,0,1,0,1,0,1,0,1]
}

df = pd.DataFrame(data)
df.head()

In [ ]:
print('Dataset shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nTarget distribution:')
print(df['Readmitted'].value_counts())

## 2. Separate Features and Target

In [ ]:
X = df.drop('Readmitted', axis=1)
y = df['Readmitted']

numeric_features = ['Age', 'Heart_Rate', 'Systolic_BP', 'Previous_Visits', 'Stay_Days']
categorical_features = ['Diagnosis']

## 3. Preprocessing Pipeline

Numerical variables are standardized and the diagnosis variable is one-hot encoded.

In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

## 4. Logistic Regression with L2 Regularization

In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        penalty='l2',
        C=1.0,
        max_iter=1000,
        random_state=42
    ))
])

## 5. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

In [ ]:
model.fit(X_train, y_train)
print('Model training completed.')

## 6. Predictions and Readmission Risk

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('Predicted classes:', y_pred)
print('\nPredicted readmission probabilities:')
print(np.round(y_prob, 3))

## 7. ROC-AUC Evaluation

In [ ]:
roc_auc = roc_auc_score(y_test, y_prob)
print(f'ROC-AUC: {roc_auc:.4f}')

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Hospital Readmission Prediction')
plt.legend()
plt.grid(True)
plt.show()

## 8. Confusion Matrix and Classification Metrics

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('Confusion Matrix:')
print(cm)

print('\nTrue Negatives :', tn)
print('False Positives:', fp)
print('False Negatives:', fn)
print('True Positives :', tp)

print('\nClassification Report:')
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Not Readmitted', 'Readmitted']
).plot()
plt.title('Confusion Matrix')
plt.show()

print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred, zero_division=0):.4f}')
print(f'ROC-AUC  : {roc_auc:.4f}')

## 9. Clinical Cost Analysis

**False Negative (FN):** The patient is actually readmitted within 30 days, but the model predicts no readmission. This may mean a missed opportunity for additional follow-up or preventive care.

**False Positive (FP):** The patient is not readmitted, but the model predicts readmission. This can lead to additional monitoring or healthcare resource use.

For a screening application, the consequences of false negatives should be considered carefully. The appropriate classification threshold should be selected using the clinical objective and the relative costs of FP and FN rather than automatically assuming a 0.5 threshold.

## 10. Threshold Analysis

In [ ]:
for threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:
    y_threshold = (y_prob >= threshold).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, y_threshold).ravel()
    recall_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) else 0

    print(
        f'Threshold={threshold:.2f} | '
        f'TP={tp_t}, FP={fp_t}, FN={fn_t}, TN={tn_t}, '
        f'Recall={recall_t:.2f}'
    )

## 11. Predict a New Patient

In [ ]:
new_patient = pd.DataFrame({
    'Age': [70],
    'Diagnosis': ['Diabetes'],
    'Heart_Rate': [102],
    'Systolic_BP': [150],
    'Previous_Visits': [5],
    'Stay_Days': [8]
})

risk_probability = model.predict_proba(new_patient)[0, 1]
print(f'Predicted 30-day readmission risk: {risk_probability:.2%}')

## 12. Conclusion

The model predicts the probability of 30-day hospital readmission using patient-level features. Logistic Regression with L2 regularization helps control model complexity, while ROC-AUC measures discrimination across classification thresholds. Confusion-matrix analysis highlights the different consequences of false negatives and false positives.

> **Note:** This notebook uses a small synthetic dataset for demonstration. It is not a clinically validated model and should not be used for real patient-care decisions.